In [6]:
from pathlib import Path
import random
from PIL import Image

# ============================================================
# Configuración
# ============================================================
base_dir = Path(".")  # cambia esto si tus carpetas no están en el directorio actual
folders = ["senior man", "young man"]

# Nombres de salida
output_names = {
    "senior man": "strip_senior_man.png",
    "young man": "strip_young_man.png",
}

# ============================================================
# Funciones
# ============================================================
def load_images_from_folder(folder_path):
    exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
    files = [p for p in folder_path.iterdir() if p.suffix.lower() in exts]
    if len(files) == 0:
        raise ValueError(f"No se encontraron imágenes en {folder_path}")
    return files

def center_crop_to_common_near_size(images):
    """
    Recorta todas las imágenes al tamaño común más grande posible
    sin deformarlas y sin redimensionar.
    
    Toma:
    - ancho común = mínimo ancho de todas
    - alto común  = mínimo alto de todas
    
    Luego hace center crop a cada imagen.
    """
    widths = [img.size[0] for img in images]
    heights = [img.size[1] for img in images]

    common_w = min(widths)
    common_h = min(heights)

    cropped = []
    for img in images:
        w, h = img.size
        
        left = (w - common_w) // 2
        top = (h - common_h) // 2
        right = left + common_w
        bottom = top + common_h
        
        cropped.append(img.crop((left, top, right, bottom)))
    
    return cropped, (common_w, common_h)

def make_horizontal_strip(folder_name, save_path):
    folder_path = base_dir / folder_name
    files = load_images_from_folder(folder_path)

    # Orden aleatorio en cada corrida
    random.shuffle(files)

    # Si quieres exactamente 4 imágenes, esto exige que haya al menos 4
    if len(files) < 4:
        raise ValueError(f"La carpeta {folder_name} tiene menos de 4 imágenes.")

    selected_files = files[:4]
    images = [Image.open(fp).convert("RGB") for fp in selected_files]

    cropped_images, (cw, ch) = center_crop_to_common_near_size(images)

    strip = Image.new("RGB", (cw * 4, ch))

    for i, img in enumerate(cropped_images):
        strip.paste(img, (i * cw, 0))

    strip.save(save_path)
    print(f"Guardada: {save_path}")
    print("Orden usado:")
    for fp in selected_files:
        print(" -", fp.name)

# ============================================================
# Ejecutar
# ============================================================
for folder in folders:
    make_horizontal_strip(folder, output_names[folder])

Guardada: strip_senior_man.png
Orden usado:
 - 4.png
 - 1.png
 - 3.png
 - 2.png
Guardada: strip_young_man.png
Orden usado:
 - 3.png
 - 1.png
 - 2.png
 - 4.png
